In [6]:
import multiprocessing as mp
from functools import partial
import pandas as pd
from tqdm import tqdm

In [7]:
from scripts._helpers import configure_logging, mute_print, set_scenario_config

In [8]:
def eurostat_per_country(input_eurostat: str, country: str) -> pd.DataFrame:
    """
    Read energy balance data for a specific country from Eurostat.

    Parameters
    ----------
    input_eurostat : str
        Path to the directory containing Eurostat data files.
    country : str
        Country code for the specific country.

    Returns
    -------
    pd.DataFrame
        Concatenated energy balance data for the specified country.

    Notes
    -----
    - The function reads `<input_eurostat>/<country>.-Energy-balance-sheets-April-2023-edition.xlsb`
    - It removes the "Cover" sheet from the data and concatenates all the remaining sheets into a single DataFrame.
    """

    filename = (
        f"{input_eurostat}/{country}-Energy-balance-sheets-April-2023-edition.xlsb"
    )
    sheet = pd.read_excel(
        filename,
        engine="pyxlsb",
        sheet_name=None,
        skiprows=4,
        index_col=list(range(4)),
        na_values=":",
    )
    sheet.pop("Cover")
    return pd.concat(sheet)

In [9]:
idees_rename = {"GR": "EL", "GB": "UK"}

In [10]:

input_eurostat ="data/eurostat/Balances-April2023"
countries = ["AT", "BE", "BG", "CH", "CY", "CZ", "DE", "DK", "EE", "ES", "FI", "FR", "HR", "HU"]
nprocesses = 1
disable_progressbar = False

"""
Return multi-index for all countries' energy data in TWh/a.

Parameters
----------
input_eurostat : str
    Path to the Eurostat database.
countries : list[str]
    List of countries for which energy data is to be retrieved.
nprocesses : int, optional
    Number of processes to use for parallel execution, by default 1.
disable_progressbar : bool, optional
    Whether to disable the progress bar, by default False.

Returns
-------
pd.DataFrame
    Multi-index DataFrame containing energy data for all countries in TWh/a.

Notes
-----
- The function first renames the countries in the input list using the `idees_rename` mapping and removes "CH".
- It then reads country-wise data using :func:`eurostat_per_country` into a single DataFrame.
- The data is reordered, converted to TWh/a, and missing values are filled.
"""

countries = {idees_rename.get(country, country) for country in countries} - {"CH"}

func = partial(eurostat_per_country, input_eurostat)
tqdm_kwargs = dict(
    ascii=False,
    unit=" country",
    total=len(countries),
    desc="Build from eurostat database",
    disable=disable_progressbar,
)
with mute_print():
    with mp.Pool(processes=nprocesses) as pool:
        dfs = list(tqdm(pool.imap(func, countries), **tqdm_kwargs))

index_names = ["country", "year", "lvl1", "lvl2", "lvl3", "lvl4"]
df = pd.concat(dfs, keys=countries, names=index_names)
df.index = df.index.set_levels(df.index.levels[1].astype(int), level=1)

# drop columns with all NaNs
unnamed_cols = df.columns[df.columns.astype(str).str.startswith("Unnamed")]
df.drop(unnamed_cols, axis=1, inplace=True)
df.drop(list(range(1990, 2022)), axis=1, inplace=True, errors="ignore")

# make numeric values where possible
df.replace("Z", 0, inplace=True)
df = df.apply(pd.to_numeric, errors="coerce")
df = df.select_dtypes(include=[np.number])

# write 'International aviation' to the lower level of the multiindex
int_avia = df.index.get_level_values(3) == "International aviation"
temp = df.loc[int_avia]
temp.index = pd.MultiIndex.from_frame(
    temp.index.to_frame().fillna("International aviation")
)
df = pd.concat([temp, df.loc[~int_avia]]).sort_index()

# Fill in missing data on "Domestic aviation" for each country.
for country in countries:
    slicer = idx[country, :, :, :, "Domestic aviation"]
    # For the Total and Fossil energy columns, fill in zeros with
    # the closest non-zero value in the year index.
    for col in ["Total", "Fossil energy"]:
        df.loc[slicer, col] = (
            df.loc[slicer, col].replace(0.0, np.nan).ffill().bfill()
        )

# Renaming some indices
index_rename = {
    "Households": "Residential",
    "Commercial & public services": "Services",
    "Domestic navigation": "Domestic Navigation",
    "International maritime bunkers": "Bunkers",
    "UK": "GB",
    "EL": "GR",
}
columns_rename = {"Total": "Total all products"}
df.rename(index=index_rename, columns=columns_rename, inplace=True)
df.sort_index(inplace=True)

# convert to TWh/a from ktoe/a
df *= 11.63 / 1e3

df

Build from eurostat database:   0%|          | 0/13 [00:00<?, ? country/s]Process SpawnPoolWorker-3:
Traceback (most recent call last):
  File "/Users/gabrieladams/miniconda3/envs/pypsa-eur/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/gabrieladams/miniconda3/envs/pypsa-eur/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/gabrieladams/miniconda3/envs/pypsa-eur/lib/python3.12/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/Users/gabrieladams/miniconda3/envs/pypsa-eur/lib/python3.12/multiprocessing/queues.py", line 389, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'eurostat_per_country' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
Process SpawnPoolWorker-2:
Traceback (most recent call last):
  File "/Users/gabrieladams/miniconda

KeyboardInterrupt: 